# 10 -- Quantum CTEM: Fully Quantum Conventional TEM Simulation

This notebook walks through QuScope's fully-quantum CTEM pipeline: **a single
quantum circuit** implements the entire imaging chain

$$|\psi_0\rangle \xrightarrow{H^{\otimes n}} |\psi_0\rangle \xrightarrow{\exp(i\sigma V)} \xrightarrow{\text{QFT}} \xrightarrow{\exp(i\chi(k))} \xrightarrow{\text{IQFT}} |\psi_{\text{image}}\rangle$$

using Qiskit `DiagonalGate` (phase grating, lens aberration function) and
`QFTGate` (the Fourier transform to/from reciprocal space) -- no classical
FFT is used in the transform steps themselves.

We'll build a 5x3 MoS2 supercell and compare:
1. **Single-slice WPOA** (weak-phase-object approximation) CTEM -- valid for thin, weakly-scattering samples
2. **Multislice CTEM** -- splits the sample into several slices with quantum Fresnel propagation between them, more accurate for thicker/stronger scatterers

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from quscope.quantum_ctem.quantum_ctem_circuit import (
    QuantumCTEMParameters, QuantumCTEMCircuit,
    relativistic_wavelength, interaction_constant,
)
from quscope.quantum_ctem.quantum_multislice_circuit import (
    QuantumMultisliceParameters, QuantumMultisliceCircuit,
)

VOLTAGE = 200e3   # 200 kV
CS_MM = 1.3       # spherical aberration, mm

# This Scherzer defocus definition can also be imported from base.py
def scherzer_defocus(cs_mm, wavelength_ang):
    '''Scherzer defocus (Angstrom), Cs-limited CTEM.'''
    return -1.2 * np.sqrt(cs_mm * 1e7 * wavelength_ang)


## Building a 5x3 MoS2 supercell potential

The atom basis (Mo at (0,0),(1/3,1/3); S at (0,1/6),(0,-1/6),(1/3,1/2),(1/3,1/6))
matches the structure factors used elsewhere in QuScope's Bloch-wave module, so
potentials built here are consistent with the dynamical-diffraction notebook (04).

The 5x3 cell works out to Lx = 5a ~= 15.9 A and Ly = 3*(a*sqrt(3)) ~= 16.5 A --
nearly square -- so we pad to a single square field of view `L = max(Lx, Ly)`
and sample it isotropically. This matters because the quantum circuits assume
one scalar `pixel_size` for both axes.

In [ ]:
def build_mos2_supercell_potential(grid_size, n_cells_x=5, n_cells_y=3):
    '''Analytic projected potential for an n_cells_x x n_cells_y MoS2
    supercell on a square, power-of-2 pixel grid. Returns (V, pixel_size).'''
    a = 3.18
    b_lat = a * np.sqrt(3.0)
    L = max(n_cells_x * a, n_cells_y * b_lat)
    px = L / grid_size

    coords = np.linspace(0.0, L, grid_size, endpoint=False)
    X, Y = np.meshgrid(coords, coords, indexing="ij")
    V = np.zeros((grid_size, grid_size))

    atom_basis = {
        "Mo": {"frac": [(0.0, 0.0), (1/3, 1/3)], "amp": 600.0, "width": 0.35},
        "S":  {"frac": [(0.0, 1/6), (0.0, -1/6), (1/3, 1/2), (1/3, 1/6)],
               "amp": 300.0, "width": 0.25},
    }
    for element, info in atom_basis.items():
        amp, width = info["amp"], info["width"]
        for fx, fy in info["frac"]:
            for icx in range(-1, n_cells_x + 1):
                for icy in range(-1, n_cells_y + 1):
                    xc, yc = (icx + fx) * a, (icy + fy) * b_lat
                    if -1.0 <= xc <= L + 1.0 and -1.0 <= yc <= L + 1.0:
                        V += amp * np.exp(-((X - xc)**2 + (Y - yc)**2) / (2 * width**2))
    return V, px


GRID_CTEM = 64
V, px = build_mos2_supercell_potential(GRID_CTEM, n_cells_x=5, n_cells_y=3)

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(V, origin="lower", cmap="viridis",
               extent=(0, GRID_CTEM*px, 0, GRID_CTEM*px))
ax.set_xlabel("x (\u00c5)"); ax.set_ylabel("y (\u00c5)")
ax.set_title("MoS2 5x3 supercell: projected potential")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()
print(f"pixel_size = {px:.4f} \u00c5/pixel, field of view = {GRID_CTEM*px:.2f} \u00c5")


## Section 1: Single-slice WPOA CTEM

The weak-phase-object approximation treats the whole sample as one thin phase
grating $t(r) = \exp(i\sigma V(r))$. We image it at (approximately) Scherzer
defocus, the "best focus" condition that maximizes contrast transfer for a
given spherical aberration $C_s$.

In [ ]:
lam = relativistic_wavelength(VOLTAGE)
defocus = scherzer_defocus(CS_MM, lam)
print(f"wavelength = {lam:.5f} \u00c5, Scherzer defocus = {defocus:.1f} \u00c5")

params = QuantumCTEMParameters(
    acceleration_voltage=VOLTAGE,
    grid_size=GRID_CTEM,
    pixel_size=px,
    defocus=defocus,
    cs=CS_MM,
)
sim = QuantumCTEMCircuit(params)
print(sim.get_info())

result_wpoa = sim.simulate(V)
print(f"circuit depth = {result_wpoa['metrics']['depth']}, "
      f"total gates = {result_wpoa['metrics']['total_gates']}")

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(result_wpoa["intensity"], cmap="gray", origin="lower",
               extent=(0, GRID_CTEM*px, 0, GRID_CTEM*px))
ax.set_title(f"Quantum CTEM (WPOA)\nC1={defocus:.0f} \u00c5, Cs={CS_MM} mm")
ax.set_xlabel("x (\u00c5)"); ax.set_ylabel("y (\u00c5)")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()


## Section 2: Quantum multislice CTEM

Multislice splits the sample into `N` thin slices along the beam direction.
Between slices, the wave is Fourier-transformed to reciprocal space (via
`QFTGate`), multiplied by a Fresnel free-space propagator (another
`DiagonalGate`), and transformed back -- genuinely alternating real-space and
reciprocal-space quantum operations, not a classical FFT under the hood.

In [ ]:
N_SLICES = 4
slice_thickness = 6.5  # ~ one MoS2 layer, Angstrom

ms_params = QuantumMultisliceParameters(
    acceleration_voltage=VOLTAGE,
    grid_size=GRID_CTEM,
    pixel_size=px,
    defocus=defocus,
    cs=CS_MM,
    slice_thickness=slice_thickness,
)
ms_sim = QuantumMultisliceCircuit(ms_params)
slices = [V / N_SLICES] * N_SLICES

result_multislice = ms_sim.simulate(slices)

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(result_multislice["intensity"], cmap="gray", origin="lower",
               extent=(0, GRID_CTEM*px, 0, GRID_CTEM*px))
ax.set_title(f"Quantum multislice CTEM ({N_SLICES} slices)")
ax.set_xlabel("x (\u00c5)"); ax.set_ylabel("y (\u00c5)")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()


## Side-by-side comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, img, title in zip(
    axes,
    [result_wpoa["intensity"], result_multislice["intensity"]],
    ["Single-slice WPOA", f"Multislice ({N_SLICES} slices)"],
):
    im = ax.imshow(img, cmap="gray", origin="lower")
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle("Quantum CTEM: WPOA vs multislice")
plt.tight_layout()
plt.show()
